In [229]:
clusterURI = "https://trd-7mb79sdg0xpkhbt4uc.z0.kusto.fabric.microsoft.com"
databaseName = "CapacityEvents"
#Workspace/Report ID
reportIDs = [("48d9a773-8469-4e19-8381-d4fdda80bdd2", "d4f8f246-5246-4e25-b415-f85bd76221b1")]
##
WSName="testLoadBalancingTarget"
CapacityIdToDeploy="df45924e-97d8-48d0-a559-3bfddbfb5aac"
TargetWorkspace=""
CUusedPercentThreshold=0.1
RecoveryThreshold=0.05
DWConnString="jm5cix3d45hellderajyxca6x4-yyfuyhctuu7uha75mtxo5wlwmm.datawarehouse.fabric.microsoft.com,1433"
DBName="WH1local"

CONFIG= {
    "clusterIngest": "https://ingest-trd-7mb79sdg0xpkhbt4uc.z0.kusto.fabric.microsoft.com",
    "clusterURL": "https://trd-7mb79sdg0xpkhbt4uc.z0.kusto.fabric.microsoft.com",
    "databaseKQL": "CapacityEvents",
    "tableKQL": "alerts",
    "tenant_id": "5f243a4b-e763-454e-ac64-88138b881ebf",
    "client_id": "1566a72f-bd94-4f99-8a38-31106058c9ee",
    "client_secret": "xxx",
    "scopeDB": "https://database.windows.net/.default",
    "scopePBI": "https://analysis.windows.net/powerbi/api/.default",
    "SQLConnString": DWConnString,
    "DBName":DBName
}

In [230]:
import requests
import os
import tempfile
from datetime import datetime
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential
import subprocess
import os
import tempfile
from datetime import datetime

from azure.kusto.data import (
    KustoConnectionStringBuilder,
    KustoClient,
    DataFormat
)
from azure.kusto.ingest import (
    QueuedIngestClient,
    IngestionProperties
)
import struct
import pyodbc
import time

In [231]:
print(WSName)
print(CapacityIdToDeploy)
print(TargetWorkspace)
print(CUusedPercentThreshold)
print(RecoveryThreshold)

testLoadBalancingTarget
df45924e-97d8-48d0-a559-3bfddbfb5aac

0.1
0.05


In [232]:
#Input checks
if CapacityIdToDeploy != "" and TargetWorkspace != "":
    raise RuntimeError("You cannot set CapacityToDeploy and TargetWorkspace at the same time")
if WSName != "" and TargetWorkspace != "":
    raise RuntimeError("You cannot set WSName and TargetWorkspace at the same time")
if CapacityIdToDeploy != "" and  WSName == "":
    raise RuntimeError("You cannot set CapacityToDeploy and not set WSName at the same time")

In [233]:
livyid="123"

def get_service_principal_token( tenant_id, client_id, client_secret, scope):
        """Fetches an OAuth2 token using a service principal."""
        credential = ClientSecretCredential(
            tenant_id=tenant_id,
            client_id=client_id,
            client_secret=client_secret
        )
        token = credential.get_token(scope)
        print(f"Token length: {len(token.token)}")
        print(f"Token (first 50 chars): {token.token[:50]}")
        return token.token

class SQLClientClass:

    def __init__(self, server, database, token):
        self.connection = self.build_connection(server, database, token)

    def build_connection(self, server, database, token):

        # Encode token as UTF-16-LE
        token_bytes = token.encode("utf-16-le")

        # Pack token length + token
        token_struct = struct.pack("<I", len(token_bytes)) + token_bytes

        connection_string = (
            "Driver={ODBC Driver 17 for SQL Server};"
            f"Server={server};"
            f"Database={database};"
            "Encrypt=yes;"
            "TrustServerCertificate=no;"
        )

        conn = pyodbc.connect(
            connection_string,
            attrs_before={1256: token_struct}  # SQL_COPT_SS_ACCESS_TOKEN
        )

        return conn


    def execute_query(self, query):
        cursor = self.connection.cursor()
        cursor.execute(query)
        return cursor


    def execute_non_query(self, query):
        cursor = self.connection.cursor()
        cursor.execute(query)
        self.connection.commit()

    def insert_log(self, message, livyid=None):
        cursor = self.connection.cursor()
    
        sql = """
        INSERT INTO logs_table (timestamp, livyid, message)
        VALUES (CURRENT_TIMESTAMP, ?, ?)
        """
    
        cursor.execute(sql, livyid, message)
        self.connection.commit()

class KustoClientClass():
    def __init__(self, cluster_url, cluster_ingest, tenant_id, client_id, client_secret, database, table):
        self.kusto_client = self.build_kusto_client(cluster_url, tenant_id, client_id, client_secret)
        self.ingest_client = self.build_ingest_client(cluster_ingest, tenant_id, client_id, client_secret)
        self.database = database
        self.table = table
    # ==========================================================
    # CONNECTION BUILDERS
    # ==========================================================
    def build_kusto_client(self, cluster, tenant_id, client_id, client_secret):
        """Creates Kusto management/query client."""
        kcsb = KustoConnectionStringBuilder.with_aad_application_key_authentication(
            cluster,
            client_id,
            client_secret,
            tenant_id
        )
        return KustoClient(kcsb)


    def build_ingest_client(self, clusterIngest, tenant_id, client_id, client_secret):
        """Creates queued ingestion client (uses ingest endpoint)."""
        ingest_cluster = clusterIngest

        ingest_kcsb = KustoConnectionStringBuilder.with_aad_application_key_authentication(
            ingest_cluster,
            client_id,
            client_secret,
            tenant_id
        )

        return QueuedIngestClient(ingest_kcsb)


    # ==========================================================
    # TABLE CREATION (Auto Create If Not Exists)
    # ==========================================================
    def ensure_table_exists(self):
        """
        Creates alerts table if it does not exist.
        """
        client = self.kusto_client

        create_command = f"""
        .create-merge table {self.table} (
            Alert: string,
            timestamp: datetime,
            Operation: string
        )
        """

        client.execute_mgmt(self.database, create_command)
        print("✅ Table ensured (created or already exists).")
    
    def getCapacityConsumption(self,capacityId,writer_instance):
        client = self.kusto_client
        torebind=False
        query = f"""
        CapacityEvents
        | extend data = parse_json(data) 
        | where todatetime(["time"]) >= ago(5m)
        | where data.capacityId == '{capacityId}'
        | project ["time"],
        CUusedPercent = todouble(data.capacityUnitMs) / todouble(data.baseCapacityUnits * 1000 * 30),
        utilizationInteractivePercent = todouble(data.utilizationInteractive)/(todouble(data.utilizationBackground) +  todouble(data.utilizationInteractive))
        """
        # synchronous call — DO NOT await
        response = client.execute(databaseName, query)
        #print(response.primary_results[0])
        max = 0
        for row in response.primary_results[0]:
            if (row[1] > max):
                max = row[1]
        writer_instance("Max capacity consumption for the last 5 mins: " + " Cap ID: " + capacityId + " Percent: " + str(max),livyid)
        return max


    # ==========================================================
    # INGESTION FUNCTION
    # ==========================================================
    def write_for_alerting(self,target_ws_cap_id, target_workspace, operation):
        """
        Creates alert record and ingests into Azure Data Explorer.
        """

        # Ensure table exists before ingestion
        self.ensure_table_exists()

        # Create alert record
        timestamp = datetime.utcnow().isoformat()
        alert_message = (
            f"Target Workspace {target_workspace} "
            f"is in an overloaded capacity {target_ws_cap_id} "
            f"- please consider moving"
        )

        csv_content = (
            "Alert,timestamp,Operation\n"
            f"{alert_message},{timestamp},{operation}\n"
        )

        # Write to temporary CSV
        with tempfile.NamedTemporaryFile(delete=False, suffix=".csv", mode="w") as tmp:
            tmp.write(csv_content)
            temp_file_path = tmp.name

        try:
            ingest_client = self.ingest_client

            ingestion_props = IngestionProperties(
                database=self.database,
                table=self.table,
                data_format=DataFormat.CSV
            )

            ingest_client.ingest_from_file(
                temp_file_path,
                ingestion_properties=ingestion_props
            )

            print("✅ Alert successfully queued for ingestion.")

        finally:
            if os.path.exists(temp_file_path):
                os.remove(temp_file_path)

def write_for_alerting(TargetWsCapID,TargetWorkspace,Op):
    # Create dummy data
    data = [(f"Target Workspace {TargetWorkspace} is in an overloaded capacity {TargetWsCapID} - please consider moving",)]

    # Create DataFrame
    df = spark.createDataFrame(data, ["Alert"])
    df = df.withColumn("timestamp", current_timestamp())

    df.write \
        .format("com.microsoft.kusto.spark.synapse.datasource") \
        .option("accessToken",accessTokenkusto) \
        .option("kustoCluster", clusterURI) \
        .option("kustoDatabase", databaseName) \
        .option("kustoTable", "alerts") \
        .option("tableCreateOptions", "CreateIfNotExist") \
        .mode("Append") \
        .save()

def getWorkspace(WSName):
    url = f"https://api.powerbi.com/v1.0/myorg/groups"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    # Check status
    #print("Status Code:", response.status_code)

    # If successful, print JSON
    if response.status_code == 200:
        data = response.json()
    
    for ws in data["value"]:
        if ws["name"] == WSName:
            return str(ws["name"])
    return 0

def GetWorkspaceId(WSName):
    url = f"https://api.powerbi.com/v1.0/myorg/groups"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    # Check status
    #print("Status Code:", response.status_code)

    # If successful, print JSON
    if response.status_code == 200:
        data = response.json()
    
    for ws in data["value"]:
        if ws["name"] == WSName:
            return str(ws["id"])
    return 0

def deployWorkspaceInCapacity(capacityId,writer_instance):
    try:
        url = f"https://api.powerbi.com/v1.0/myorg/groups"
        headers = {
            "Authorization": f"Bearer {accessTokenpbi}",
            "Content-Type": "application/json"
        }
        payload = {
            "name": WSName #hardcoded
        }
        response = requests.post(url, headers=headers,json=payload)
        # Check status
        #print("Status Code:", response.status_code)

        # If successful, print JSON
        if response.status_code == 200:
            dataws = response.json()
        #print(dataws["id"])

        url = f"https://api.powerbi.com/v1.0/myorg/admin/capacities/AssignWorkspaces"
        headers = {
            "Authorization": f"Bearer {accessTokenpbi}",
            "Content-Type": "application/json"
        }
        payload = {
            "capacityMigrationAssignments":[
                {
                "targetCapacityObjectId": capacityId,
                "workspacesToAssign": [dataws["id"]]
                }
            ]
        }
        response = requests.post(url, headers=headers,json=payload)
        # Check status
        #print("Status Code: ", response.text)
        # If successful, print JSON
    except Exception as e:
        #print("E", e)
        writer_instance( "Could not create workspace: " + str(dataws["id"]) + " and attach it to capacity " + capacityId ,livyid)
        return "0"
    return str(dataws["id"])
#

###############

def get_sem_model_name(datasetID):
    #pending optimization for pagination
    url = f"https://api.fabric.microsoft.com/v1/workspaces"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    # Check status
    #print("Status Code:", response.status_code)

    # If successful, print JSON
    if response.status_code == 200:
        data = response.json()
    for i in data["value"]:
        workspaceid= (i["id"])
        url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspaceid}/datasets"
        headers = {
            "Authorization": f"Bearer {accessTokenpbi}",
            "Content-Type": "application/json"
        }
        response = requests.get(url, headers=headers)
        # If successful, print JSON
        if response.status_code == 200:
            data = response.json()
        for datasets in data["value"]:
            if str(datasets["id"]).strip() == datasetID:
                return (str(datasets["name"]))
    return "0"

##############################


def get_sem_model_ID(datasetName,workspaceId):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspaceId}/datasets"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)

    # If successful, print JSON
    if response.status_code == 200:
        data = response.json()
    for i in data["value"]:
        if i["name"]==datasetName:
            return i["id"]

def rebind_report(groupId,reportId,datasetID):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{groupId}/reports/{reportId}/Rebind"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    payload = {
        "datasetId": datasetID
    }
    response = requests.post(url, headers=headers,json=payload)

def get_workspaceId_from_dataset(datasetID):
    #pending optimization for pagination
    url = f"https://api.fabric.microsoft.com/v1/workspaces"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    # Check status
    #print("Status Code:", response.status_code)

    # If successful, print JSON
    if response.status_code == 200:
        data = response.json()
    for i in data["value"]:
        workspaceid= (i["id"])
        url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspaceid}/datasets"
        headers = {
            "Authorization": f"Bearer {accessTokenpbi}",
            "Content-Type": "application/json"
        }
        response = requests.get(url, headers=headers)
        # If successful, print JSON
        if response.status_code == 200:
            data = response.json()
        for datasets in data["value"]:
            if str(datasets["id"]).strip() == datasetID:
                return (str(workspaceid))

def get_workspaceId_from_report(reportId):
    #pending optimization for pagination
    url = f"https://api.fabric.microsoft.com/v1/workspaces"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    # If successful, print JSON
    if response.status_code == 200:
        data = response.json()
    for i in data["value"]:
        workspaceid= (i["id"])
        url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspaceid}/reports"
        headers = {
            "Authorization": f"Bearer {accessTokenpbi}",
            "Content-Type": "application/json"
        }
        response = requests.get(url, headers=headers)
        # If successful, print JSON
        if response.status_code == 200:
            data = response.json()
        for reports in data["value"]:
            if str(reports["id"]).strip() == reportId:
                return (str(workspaceid))


def get_capacity_id_from_workspace(workspaceId):
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    # Check status
    #print("Status Code:", response.status_code)

    # If successful, print JSON
    #print(response.text)
    if response.status_code == 200:
        data = response.json()
    return data["capacityId"]


def rebind_datasources(workspaceId,datasetID,datasourceId,connectivityType,path,type,gatewayId):
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/semanticModels/{datasetID}/bindConnection"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    payload = {
    "connectionBinding": {
    "id": datasourceId,
    "connectivityType": connectivityType,
    "connectionDetails": {
        "path": path,
        "type": type
        }
      }
    }
    # ✅ Add gatewayId only if provided
    if gatewayId:
        payload["connectionBinding"]["gatewayId"] = gatewayId
    
    response = requests.post(url, headers=headers,json=payload)
    # Check status
    #print("response.text")
    #print(response.text)
    #print("response.text")


def getConnectionDetailsFromDatasetandRebind(SRCworkspaceID,SRCdatasetId,workspaceID,datasetId,writer_instance):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{SRCworkspaceID}/datasets/{SRCdatasetId}/datasources"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    
    response = requests.get(url, headers=headers)
    # Check status
    #print("Status Code:", response.status_code)
    #print(response.text)
    # If successful, print JSON
    if response.status_code == 200:
        data = response.json()
    for i in data["value"]:
        #print(i["datasourceId"])
        connectionId=str(i["datasourceId"]).strip()
        url = f"https://api.fabric.microsoft.com/v1/connections/{connectionId}"
        headers = {
            "Authorization": f"Bearer {accessTokenpbi}",
            "Content-Type": "application/json"
        }        
        response = requests.get(url, headers=headers)
        #print(response.text)
        if response.status_code == 200:
            data1 = response.json()
        datasourceId = i["datasourceId"]
        connectivityType = data1["connectivityType"]
        gatewayId = data1.get("gatewayId")
        path = data1["connectionDetails"]["path"]
        type = data1["connectionDetails"]["type"]
        writer_instance("Rebinding connection at dataset " + datasetId + " datasourceId " + datasourceId + " path: " + path ,livyid)
        rebind_datasources(workspaceID,datasetId,datasourceId,connectivityType,path,type,gatewayId)

def reportRebind(dictRebinds,writer_instance,SQLClientClass):
    #Rebind reports to target dataset 
    for report in dictRebinds:
        SQLClientClass.insert_log("Initial DS ID "+ str(report["InitialDatasetID"]) +" Report WS: " +str(report["report_Workspace_id"]) + " REPORT: "+ str(report["report_id"]) + " repointed at "  + str(report["trgdatasetID"]) ,livyid)
        if report["ToRebind"]>0:
            rebind_report(str(report["report_Workspace_id"]).strip(), str(report["report_id"]).strip(), str(report["trgdatasetID"]).strip())
            #print("!writing to table!")
            if report["InitialDatasetID"]!="CALLBACK":
                values_list = []
                values_list.append(f"('{report['InitialDatasetID']}', '{report['report_Workspace_id']}', "f"'{report['report_id']}', '{report['trgdatasetID']}', '{report['TargetWsID']}')")
                values_clause = ",\n".join(values_list)              
                # Construct the dynamic MERGE SQL
                merge_sql = f"""
                MERGE INTO report_repoints AS t
                USING (
                    VALUES
                    {values_clause}
                ) AS s (InitialDatasetID, report_Workspace_id, report_id, trgdatasetID, TargetWsID)
                ON t.report_id = s.report_id
                WHEN MATCHED THEN
                    UPDATE SET
                        t.report_Workspace_id = s.report_Workspace_id,
                        t.trgdatasetID = s.trgdatasetID,
                        t.TargetWsID = s.TargetWsID,
                        t.created_at = SYSDATETIME()
                WHEN NOT MATCHED THEN
                    INSERT (
                        InitialDatasetID,
                        report_Workspace_id,
                        report_id,
                        trgdatasetID,
                        TargetWsID,
                        created_at
                    )
                    VALUES (
                        s.InitialDatasetID,
                        s.report_Workspace_id,
                        s.report_id,
                        s.trgdatasetID,
                        s.TargetWsID,
                        SYSDATETIME()
                    );
                """

                # Execute
                SQLClientClass.execute_non_query(merge_sql)

def update_refresh_schedule(sourceWS, sourceDatasetID, TargetWS,TargetDatasetID):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{sourceWS}/datasets/{sourceDatasetID}/refreshSchedule"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
            data1 = response.json()
    #print(data1)
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    payload1 = {
        "value": {
            "enabled": True,
            "days": data1["days"],
            "times": data1["times"],
            "localTimeZoneId": data1["localTimeZoneId"]
        }
    }
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{TargetWS}/datasets/{TargetDatasetID}/refreshSchedule"
    response = requests.patch(url, headers=headers,json=payload1)
    #print("##")
    #print(response)
    #print("##")

def get_refresh_schedule(sourceWS, sourceDatasetID):
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{sourceWS}/datasets/{sourceDatasetID}/refreshSchedule"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
            data1 = response.json()
    #print(data1)
    

def get_workspace_name(workspace_id: str) -> str:
    """
    Retrieve the workspace (group) name from the Power BI / Fabric API.

    Args:
        workspace_id: The workspace (group) ID.
        access_token: A valid OAuth2 access token with Workspace.Read.All scope.

    Returns:
        The workspace name as a string.
    """

    
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}"
    #print(url)

    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }

    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        raise Exception(f"Failed to get workspace info: {response.status_code} - {response.text}")
    
    data = response.json()
    return data["name"]  # this is the workspace name

def deploySemanticModel(sourceWorkspaceName,targetWorkspaceName,datasetName,writer_instance):
    # Model names
    model_name = datasetName
    model_name_target = model_name + "_LB"

    # 1️⃣ Create a dynamic export directory inside the current working directory
    cwd = os.getcwd()
    export_dir = os.path.join(cwd, "exports")
    os.makedirs(export_dir, exist_ok=True)  # creates 'exports' if it doesn't exist

    # Path for exported semantic model
    export_file_path = os.path.join(export_dir, f"{model_name}.SemanticModel")

    # 2️⃣ Run export
    export_cmd = [
        "fab",
        "export",
        f"{sourceWorkspaceName}.Workspace/{model_name}.SemanticModel",
        "-o",
        export_dir,
        "-f"
    ]

    writer_instance(f"Exporting model '{model_name}' from workspace '{sourceWorkspaceName}'...")
    subprocess.run(export_cmd, check=True)

    # 3️⃣ Run import
    import_cmd = [
        "fab",
        "import",
        f"{targetWorkspaceName}.Workspace/{model_name_target}.SemanticModel",
        "--input",
        export_file_path,
        "-f"
    ]

    writer_instance(f"Importing model '{model_name_target}' into workspace '{targetWorkspaceName}'...")
    subprocess.run(import_cmd, check=True)

    writer_instance("Export and import completed successfully.")

def refreshDataset(workspaceId,datasetId):
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspaceId}/datasets/{datasetId}/refreshes"
    response = requests.post(url, headers=headers)
    #print("##")
    #print(response.status_code)
    #print("##")

In [234]:
#get tokens
accessTokenDBs = get_service_principal_token(CONFIG["tenant_id"], CONFIG["client_id"], CONFIG["client_secret"], CONFIG["scopeDB"])
accessTokenpbi = get_service_principal_token(CONFIG["tenant_id"], CONFIG["client_id"], CONFIG["client_secret"], CONFIG["scopePBI"])
SQLClientClass = SQLClientClass(CONFIG["SQLConnString"], CONFIG["DBName"], accessTokenDBs)
kustoClientInstance = KustoClientClass(CONFIG["clusterURL"], CONFIG["clusterIngest"], CONFIG["tenant_id"], CONFIG["client_id"],CONFIG["client_secret"], CONFIG["databaseKQL"], CONFIG["tableKQL"])

os.environ['FAB_TOKEN'] = accessTokenpbi
os.environ['FAB_TOKEN_ONELAKE'] = accessTokenpbi

Token length: 1696
Token (first 50 chars): eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6InNNMV
Token length: 1624
Token (first 50 chars): eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6InNNMV


In [235]:
SQLClientClass.execute_non_query("""
IF NOT EXISTS (
    SELECT * 
    FROM sys.tables 
    WHERE name = 'report_repoints'
)
BEGIN
    CREATE TABLE report_repoints (
        InitialDatasetID VARCHAR(255),
        report_Workspace_id VARCHAR(255),
        report_id VARCHAR(255),
        trgdatasetID VARCHAR(255),
        TargetWsID VARCHAR(255),
        created_at DATETIME2(3)
    );
END
""")

SQLClientClass.execute_non_query("""
IF NOT EXISTS (
    SELECT * 
    FROM sys.tables 
    WHERE name = 'logs_table'
)
BEGIN
CREATE TABLE logs_table (
    timestamp DATETIME2(3),
    livyid VARCHAR(max),
    message VARCHAR(255)
);
END""")

In [236]:
#does ws exist already?
targetWSCapacityId=""
if WSName!="": #in case the name is given for new deployment
    existingwsid=GetWorkspaceId(WSName)
    if existingwsid!=0: #does workspace exist
        try:
            targetWSCapacityId=str(get_capacity_id_from_workspace(existingwsid)).strip()
        except Exception as e:
            print(e)
            raise Exception("Make sure workspace " + existingwsid + " is in a fabric capacity")
if TargetWorkspace!="":
    try:
        targetWSCapacityId=str(get_capacity_id_from_workspace(TargetWorkspace)).strip()
    except Exception as e:
        print(e)
        raise Exception("Make sure workspace " + TargetWorkspace + " is in a fabric capacity")

In [237]:
#Get report dependencies
if TargetWorkspace!="":
    TargetWorkspace = TargetWorkspace #no op, just for completion
else: 
    if existingwsid==0: #does not exist
        if CapacityIdToDeploy != "":
            SQLClientClass.insert_log("Deploying workspace at Capacity " + " workspace: " + WSName + " capacity: " + CapacityIdToDeploy,livyid)
            TargetWorkspace = deployWorkspaceInCapacity(CapacityIdToDeploy,SQLClientClass.insert_log)
            print("deploying workspace " + WSName + " in capacity " + CapacityIdToDeploy)
            time.sleep(10)
    else:
        TargetWorkspace = existingwsid.strip()
print("TargetWorkspace: " , TargetWorkspace)


if targetWSCapacityId=="":
    TargetWsCapID = str(get_capacity_id_from_workspace(TargetWorkspace)).strip() #in case it was deployed now
else:
    TargetWsCapID=targetWSCapacityId

dictRebinds=[]
ReportsRebindBack=[]
print("reportIDs")
print(reportIDs)
for workspace_id,report_id in reportIDs:
    reportWS=get_workspaceId_from_report(report_id)
    SQLClientClass.insert_log("CHECKING FOR REPORT " + str(report_id)+ " at workspace "+ reportWS,livyid)

    #row = (
    #    spark.table("report_repoints")
    #    .filter(col("report_id") == report_id)
    #    .select("*")
    #    .limit(1)
    #    .collect()
    #)

    result = SQLClientClass.execute_query(f"SELECT * FROM report_repoints where report_id = '{report_id}'")
    row = result.fetchone()
    # Convert to dict: column -> value
    if row:
        columns = [column[0] for column in result.description]  # column names
        row_dict = dict(zip(columns, row))

        #what happens if the dataset is published again? does it update ID?
        #Posso testar no caso de fazer dois rebinds para dois workspaces diferentes?
        print("ROW " , row_dict)
        initial_ds_id = row_dict.get("InitialDatasetID")
        Init_Workspace_id = get_workspaceId_from_dataset(initial_ds_id)
        InitialCapId=get_capacity_id_from_workspace(str(Init_Workspace_id))
        InitCapCons=kustoClientInstance.getCapacityConsumption(InitialCapId,SQLClientClass.insert_log)
        if InitCapCons < RecoveryThreshold:
            #Dont copy as I assume the dataset is already updated via CICD in initial workspace (only rebind report)
            dictRebinds.append({
                "InitialDatasetID":"CALLBACK",
                "TargetWsID":Init_Workspace_id,
                "ToRebind":1,
                "report_Workspace_id": reportWS,
                "report_id": report_id,
                "trgdatasetID": initial_ds_id,
            })
            #######################
            SQLClientClass.execute_non_query(f"DELETE FROM report_repoints WHERE report_id = '{report_id}'")
            #######################
            SQLClientClass.insert_log("REPORT " + report_id + " WHICH WAS PREVIOUSLY REPOINTED AT DATASET " + str(row_dict.get("trgdatasetID")) + " WILL BE REPOINTED BACK AT " + initial_ds_id ,livyid)
            ReportsRebindBack.append(report_id)
    else:
        SQLClientClass.insert_log("Report: " + report_id + " not in cache for repoint to original" ,livyid)

reportRebind(dictRebinds,SQLClientClass.insert_log,SQLClientClass)
#from now we only want to iterate over reports which were not repointed back already (as those ones are definitely healthy)
ReportsRemaining = [(ws_id,rp_id) for ws_id,rp_id in reportIDs if rp_id not in ReportsRebindBack]
#clean up dictRebinds
dictRebinds=[]
print("ReportsRemaining")
print(ReportsRemaining)
for workspace_id,report_id in ReportsRemaining:
    #If target workspace is in an overloaded capacity we can raise this exception and exit
    Op = kustoClientInstance.getCapacityConsumption(TargetWsCapID,SQLClientClass.insert_log)
    if Op > CUusedPercentThreshold*0.8:
        SQLClientClass.insert_log("NO OP - The target workspace for Load balancing is already in an overloaded capacity ( > 0.8 * CUusedPercentThreshold ) " + " CAPACITY: " + TargetWsCapID + " WORKSPACE: " + TargetWorkspace,livyid)
        kustoClientInstance.write_for_alerting(TargetWsCapID,TargetWorkspace,Op)
        raise RuntimeError("NO OP - The target workspace for Load balancing is already in an overloaded capacity ( > 0.8 * CUusedPercentThreshold ) " + " CAPACITY: " + TargetWsCapID + " WORKSPACE: " + TargetWorkspace)
        #notebookutils.notebook.exit("NO OP - The target workspace for Load balancing is already in an overloaded capacity")
        #raise RuntimeError("NO OP - The target workspace for Load balancing is already in an overloaded capacity")
    ##############

    url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/reports/{report_id}"
    headers = {
        "Authorization": f"Bearer {accessTokenpbi}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    print(response.status_code)
    if response.status_code == 200:
        data = response.json()
        datasetName=get_sem_model_name(data["datasetId"])
        Workspace_id = get_workspaceId_from_dataset(data["datasetId"])
        report_workspace_id=reportWS
        Capacity_id = str(get_capacity_id_from_workspace(str(Workspace_id))).strip()

        #for optimization only in case TargetWsCapID <= CUusedPercentThreshold - as it was already measured above dont need to run ToRebind=getCapacityConsumption(Capacity_id)
        if Capacity_id == TargetWsCapID:
            SQLClientClass.insert_log("The report is in workspace Id " + str(Workspace_id) + " which is in the same capacity of the target workspace " + str(TargetWorkspace) ,livyid)
            continue
            
        print("Capacity ID")
        print(Capacity_id)
        ToRebind=kustoClientInstance.getCapacityConsumption(Capacity_id,SQLClientClass.insert_log)
        if ToRebind>CUusedPercentThreshold:
            try:
                #labs.deploy_semantic_model(data["datasetId"],Workspace_id,target_dataset=datasetName+"_LB",target_workspace=TargetWorkspace, refresh_target_dataset =False, overwrite =True)
                sourceWorkspaceName=get_workspace_name(Workspace_id)
                targetWorkspaceName=get_workspace_name(TargetWorkspace)
                deploySemanticModel(sourceWorkspaceName,targetWorkspaceName,datasetName,SQLClientClass.insert_log)
                trgdatasetID=get_sem_model_ID(datasetName+"_LB",TargetWorkspace)
            except Exception as e:
                #this following line is to get target dataset ID in case the refresh of the semantic model fails (due to datasource non-configurations)
                trgdatasetID=get_sem_model_ID(datasetName+"_LB",TargetWorkspace)
                print(f"Exception (may not be fatal): {e}")
            getConnectionDetailsFromDatasetandRebind(Workspace_id,str(data["datasetId"]).strip(),TargetWorkspace,trgdatasetID,SQLClientClass.insert_log)
            update_refresh_schedule(Workspace_id, str(data["datasetId"]).strip(), TargetWorkspace ,trgdatasetID)
            refreshDataset(datasetId=trgdatasetID, workspaceId=TargetWorkspace)
            dictRebinds.append({
                "InitialDatasetID":data["datasetId"],
                "TargetWsID":TargetWorkspace,
                "ToRebind":ToRebind,
                "report_Workspace_id": report_workspace_id,
                "report_id": report_id,
                "trgdatasetID": trgdatasetID,
            })
        else:
            SQLClientClass.insert_log("NOT REBINDING REPORT: " + str(report_id) + " LIKELY BECAUSE CAPACITY UTILIZATION IS LOW",livyid)

reportRebind(dictRebinds,writer_instance,SQLClientClass)

TargetWorkspace:  edccd36a-eef3-4ef8-b90f-6a80afeb54d2
reportIDs
[('48d9a773-8469-4e19-8381-d4fdda80bdd2', 'd4f8f246-5246-4e25-b415-f85bd76221b1')]
ReportsRemaining
[('48d9a773-8469-4e19-8381-d4fdda80bdd2', 'd4f8f246-5246-4e25-b415-f85bd76221b1')]
200
Capacity ID
df45924e-97d8-48d0-a559-3bfddbfb5aac
